# DIRE Table 5 — LSUN-Bedroom, one-run Kaggle workflow

Notebook tự chứa toàn luồng **index → ghép triplet → subset (tuỳ chọn) → train 4 representation → test → report**. Sau khi cấu hình các input roots và bật xác nhận GPU ở cell cấu hình, có thể chọn **Run All**; không cần chuyển notebook/stage qua nhiều lượt.

## Phạm vi và quyết định dữ liệu

- Chỉ LSUN-Bedroom/Table 5; không kéo ImageNet, CelebA-HQ hay toàn bộ generator train data.
- Dùng `images`, `recons`, `dire` đã mount trên Kaggle, đọc-only; pipeline không sao chép cả dataset và tạo input tensor theo batch.
- Protocol Table-5-style trong repo cần `train/val` cho `real + adm`, và `test` cho `real + adm + ddpm + iddpm + pndm + sdv2`. Các generator ngoài ADM chỉ cần test. Mặc định chọn **full matched intersection** của các mẫu có đủ 3 artifact. Chế độ pilot chọn tối đa N ID/cell bằng hash ổn định; cùng ID được dùng cho cả images/recons/dire và cả bốn representation. Nếu đối chiếu một protocol ngoài repo yêu cầu SD-v1, phải có cell SD-v1 thật và cấu hình riêng; không thể thay bằng SD-v2.
- Dataset LSUN-Bedroom hiện có trong workspace dùng `sdv2`; mặc định notebook ghi rõ generator này là SD-v2 và **không gọi là SD-v1**. Nếu một upload khác có SD-v1 thật, sửa cell cấu hình để dùng `sdv1`. Bảng chỉ dùng SD-v2 phải được mô tả là kết quả theo dữ liệu hiện có/biến thể Table-5-style, không phải cột SD-v1.
- Các thư mục lặp tên lá như `lsun_bedroom/lsun_bedroom` hoặc `real/real` được xử lý bằng đường dẫn chuẩn hoá trong bộ index; input không bị rename/copy.

## Giới hạn protocol phải biết

Notebook dùng DIRE maps/recons phát hành, nên tự ghi mức **R2-style: detector train/eval mới trên artifact đầu vào có sẵn**; nó không tính lại diffusion maps. Dấu `RGB_DIRE` được hiện thực bằng nối kênh RGB rồi DIRE và ImageNet conv1 được mở rộng bằng cách lặp/chia đôi trọng số; paper/code tác giả chưa xác nhận rõ chính xác fusion này. Chiều scale DIRE đọc theo PNG như ảnh `[0,1]`; metadata ghi cấu hình này. Nếu map trên Kaggle có encoding khác, dừng và đổi cấu hình có chủ đích — không đoán scale.

Đây là pipeline đầy đủ, nhưng **agent không chạy notebook hay khởi động GPU**. Chạy toàn bộ 4 detector là chi phí compute của bạn; `CONFIRM_GPU_RUN` mặc định tắt để tránh vô tình kích hoạt khi chỉ kiểm tra notebook.


In [ ]:
# Stage 0 — cấu hình duy nhất cho cả luồng
from pathlib import Path
import os, json, hashlib, random, csv, time, math, platform, sys

# Nếu chỉ có một bộ Dataset chứa asset con images/recons/dire, có thể để None để dò
# thư mục con cấp 1. Nếu có nhiều kết quả trùng tên, khai báo đường dẫn tuyệt đối.
INPUT_ROOTS = {
    "images": None,  # ví dụ: /kaggle/input/diffusionforensics-images/images
    "recons": None,  # ví dụ: /kaggle/input/diffusionforensics-recons/recons
    "dire": None,    # ví dụ: /kaggle/input/diffusionforensics/dire
}
IMAGENET_CHECKPOINT = None  # path tới torchvision ResNet-50 ImageNet state_dict (.pth)
DATASET_VERSION_TAG = "lsun-table5-input-v1"  # đổi tag nếu thay version input Kaggle
DOMAIN = "lsun_bedroom"
RUN_ID = "table5-lsun-2026"

# Chế độ chính được chọn: full các sample IDs khớp trong cell paper.
# Đặt RUN_MODE="pilot" và PILOT_PER_CELL=200 chỉ để smoke/pilot; không phải kết quả paper.
RUN_MODE = "full"                 # "full" | "pilot"
PILOT_PER_CELL = 200
SEED = 3407
# Current LSUN-Bedroom upload uses SD-v2. This is explicit, not an SD-v1 alias.
# Only change to sdv1 if the mounted dataset actually contains SD-v1 samples.
TEST_GENERATORS = ["adm", "ddpm", "iddpm", "pndm", "sdv2"]
PROTOCOL_LABEL = "table5_style_lsun_bedroom_sdv2_data_variant_not_sdv1"

# Dùng PNG DIRE đã phát hành như ảnh: PIL -> ToTensor [0,1]. Chỉ đổi nếu có
# chứng cứ metadata cho biết map được lưu ở encoding float/raw khác.
DIRE_DECODE_MODE = "published_png_to_tensor_0_1"
REPRESENTATIONS = ["REC", "RGB", "RGB_DIRE", "DIRE"]

# Nguồn tham chiếu code tác giả: ResNet-50 ImageNet, Adam lr=1e-4, bs=64,
# 400 epoch tối đa; validation ACC/patience và threshold strict >0.5.
MAX_EPOCHS = 400
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
PATIENCE = 5
NUM_WORKERS = 4
SAVE_RESUME_EVERY = 5
CONFIRM_GPU_RUN = False  # đặt True sau khi đọc inventory + chốt protocol

OUTPUT_ROOT = Path("/kaggle/working") / f"dire-{RUN_ID}"
MANIFEST_PATH = OUTPUT_ROOT / "triplet_manifest.json"
SELECTED_PATH = OUTPUT_ROOT / "selected_ids.json"
RESULTS_PATH = OUTPUT_ROOT / "results"
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}

assert RUN_MODE in {"full", "pilot"}
assert DOMAIN == "lsun_bedroom"
assert len(TEST_GENERATORS) == 5 and len(set(TEST_GENERATORS)) == 5
assert DIRE_DECODE_MODE == "published_png_to_tensor_0_1"
print({"run_id": RUN_ID, "run_mode": RUN_MODE, "protocol": PROTOCOL_LABEL,
       "test_generators": TEST_GENERATORS, "training_confirmed": CONFIRM_GPU_RUN,
       "output_root": str(OUTPUT_ROOT)})

## Stage 1 — resolve the three mounted inputs

Chỉ xem thư mục cấp cao và các nhánh LSUN cần thiết, không decode/hash lại toàn bộ ảnh. Nếu Kaggle mount nhiều thư mục cùng tên, điền ba đường dẫn ở trên thay vì để notebook tự chọn.

In [ ]:
# Stage 1 — resolve roots + normalized path indexing helpers
from collections import defaultdict

KAGGLE_INPUT = Path("/kaggle/input")


def resolve_asset_root(asset: str, explicit):
    if explicit:
        root = Path(explicit).expanduser().resolve()
        if not root.is_dir():
            raise FileNotFoundError(f"{asset} root không tồn tại: {root}")
        return root
    if not KAGGLE_INPUT.is_dir():
        raise FileNotFoundError("Không thấy /kaggle/input; notebook này cần chạy trong Kaggle.")
    candidates = set()
    for dataset_root in KAGGLE_INPUT.iterdir():
        if not dataset_root.is_dir():
            continue
        if dataset_root.name.lower() == asset:
            candidates.add(dataset_root.resolve())
        child = dataset_root / asset
        if child.is_dir():
            candidates.add(child.resolve())
    if len(candidates) != 1:
        raise RuntimeError(
            f"Không thể tự chọn root {asset}: {sorted(map(str, candidates))}. "
            f"Điền INPUT_ROOTS['{asset}'] bằng path chính xác."
        )
    return next(iter(candidates))


ROOTS = {asset: resolve_asset_root(asset, INPUT_ROOTS[asset]) for asset in INPUT_ROOTS}
for asset, root in ROOTS.items():
    print(f"{asset:>6}: {root}")


def collapse_duplicate_components(parts):
    """Remove only consecutive repeated names; never mutate source paths."""
    out = []
    for part in parts:
        part = str(part)
        if not out or part != out[-1]:
            out.append(part)
    return tuple(out)


def canonical_generator(name):
    name = str(name).lower()
    if name == "sdv1_test":
        return "sdv1"
    return name


def has_images(path):
    if not path.is_dir():
        return False
    for current, _, files in os.walk(path):
        if any(Path(name).suffix.lower() in IMAGE_SUFFIXES for name in files):
            return True
    return False


def resolve_domain_root(asset_root, split, relevant_generators):
    outer = asset_root / split / DOMAIN
    candidates = [outer, outer / DOMAIN]
    hits = []
    for candidate in candidates:
        if not candidate.is_dir():
            continue
        if any((candidate / g).is_dir() for g in relevant_generators):
            hits.append(candidate)
    if not hits:
        raise FileNotFoundError(f"Không thấy {split}/{DOMAIN} dưới {asset_root}")
    if len(hits) > 1:
        raise RuntimeError(f"Layout domain mơ hồ (có cả leaf trực tiếp/lặp): {hits}")
    return hits[0]


def resolve_generator_root(domain_root, generator):
    aliases = [generator]
    if generator == "sdv1":
        aliases.append("sdv1_test")
    candidates = [domain_root / name for name in aliases]
    found = [p for p in candidates if p.is_dir()]
    if len(found) != 1:
        raise FileNotFoundError(f"Thiếu hoặc trùng folder generator {generator}: {found}")
    return found[0]


def select_content_leaf(generator_root, generator_name):
    leaf = generator_root
    while True:
        repeated = leaf / leaf.name
        direct_images = any(p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES
                            for p in leaf.iterdir())
        if not repeated.is_dir():
            return leaf
        if direct_images:
            raise RuntimeError(f"Layout mơ hồ: có ảnh ở cả {leaf} và {repeated}")
        # Follow consecutive repeated-name directories without crawling their contents.
        leaf = repeated


def scan_cell(asset_root, split, generator, relevant_generators):
    domain_root = resolve_domain_root(asset_root, split, relevant_generators)
    generator_root = resolve_generator_root(domain_root, generator)
    leaf = select_content_leaf(generator_root, generator_root.name)
    found = {}
    for current, dirs, files in os.walk(leaf):
        dirs.sort()
        for name in sorted(files):
            path = Path(current) / name
            if path.suffix.lower() not in IMAGE_SUFFIXES:
                continue
            rel = path.relative_to(leaf).with_suffix("").as_posix()
            # Normalization is for identity only. The underlying Kaggle path is untouched.
            rel = "/".join(collapse_duplicate_components(rel.split("/")))
            if rel in found:
                raise RuntimeError(f"ID trùng sau chuẩn hoá ở {split}/{generator}: {rel}")
            size = path.stat().st_size
            if size <= 0:
                raise RuntimeError(f"File rỗng trong cell bắt buộc: {path}")
            found[rel] = {"path": str(path), "size_bytes": size}
    if not found:
        raise RuntimeError(f"Cell rỗng: {asset_root}/{split}/{DOMAIN}/{generator}")
    return found


def required_cells(test_generators):
    cells = [("train", "real"), ("train", "adm"), ("val", "real"), ("val", "adm"),
             ("test", "real")]
    cells.extend(("test", g) for g in test_generators)
    return cells

print("Index helpers ready; repeated domain/class leaves are mapped, never copied.")

## Stage 2 — build one matched triplet manifest

Khóa lựa chọn theo `split/domain/generator/sample_id`, ghép bằng đường dẫn tương đối bên trong leaf đã chuẩn hoá. Không dùng basename counter. Nếu cùng một ID thiếu ở một modality hoặc xuất hiện lặp sau chuẩn hoá, notebook dừng và in chênh lệch; không tự âm thầm loại ảnh để làm đẹp counts.

In [ ]:
# Stage 2 — metadata-only scan of required LSUN cells, then strict triplet join

def build_triplet_manifest():
    cells = required_cells(TEST_GENERATORS)
    rows, cell_counts, mismatch_notes = [], {}, []
    for split, generator in cells:
        per_asset = {
            asset: scan_cell(ROOTS[asset], split, generator,
                             [g for _, g in cells if _ == split])
            for asset in ("images", "recons", "dire")
        }
        sets = {asset: set(records) for asset, records in per_asset.items()}
        common = set.intersection(*sets.values())
        union = set.union(*sets.values())
        missing = {asset: sorted(union - ids) for asset, ids in sets.items() if union - ids}
        cell = f"{split}/{DOMAIN}/{generator}"
        cell_counts[cell] = {asset: len(ids) for asset, ids in sets.items()}
        if missing:
            mismatch_notes.append({"cell": cell, "counts": cell_counts[cell],
                                   "missing_ids_preview": {k: v[:5] for k, v in missing.items()},
                                   "n_missing": {k: len(v) for k, v in missing.items()}})
            raise RuntimeError(
                f"Triplet mismatch in {cell}: {cell_counts[cell]}; "
                f"missing preview={ {k:v[:3] for k,v in missing.items()} }. "
                "Notebook will not silently drop unmatched IDs."
            )
        for sample_id in sorted(common):
            row = {"split": split, "domain": DOMAIN, "generator": generator,
                   "label": 0 if generator == "real" else 1, "sample_id": sample_id}
            for asset in ("images", "recons", "dire"):
                row[f"{asset}_path"] = per_asset[asset][sample_id]["path"]
                row[f"{asset}_bytes"] = per_asset[asset][sample_id]["size_bytes"]
            rows.append(row)
        print(f"{cell:32s} matched={len(common):,} per_asset={cell_counts[cell]}")
    return rows, cell_counts


OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if OUTPUT_ROOT.resolve() in [root.resolve() for root in ROOTS.values()]:
    raise RuntimeError("Output không được trùng với mounted input")
all_triplets, raw_counts = build_triplet_manifest()
manifest_payload = {
    "schema": "lsun-table5-matched-triplets-v1",
    "dataset_version_tag": DATASET_VERSION_TAG,
    "protocol_label": PROTOCOL_LABEL,
    "roots": {k: str(v) for k, v in ROOTS.items()},
    "normalization": "strip consecutive duplicate directory component from path identity only",
    "cells": raw_counts,
    "rows": all_triplets,
}
canonical = json.dumps(manifest_payload, sort_keys=True, separators=(",", ":")).encode()
manifest_payload["manifest_sha256"] = hashlib.sha256(canonical).hexdigest()
MANIFEST_PATH.write_text(json.dumps(manifest_payload, indent=2), encoding="utf-8")
print("Matched manifest:", MANIFEST_PATH, "rows=", len(all_triplets),
      "id_manifest_sha256=", manifest_payload["manifest_sha256"])


## Stage 3 — choose full or matched pilot

**Khuyến nghị cho kết quả chính:** `RUN_MODE="full"` trên toàn bộ matched IDs của các cell LSUN Table 5. Đây là tập con theo *protocol* (một domain, train real+ADM, test các generator cần thiết), không phải toàn bộ DiffusionForensics. Không cần giảm mỗi modality riêng; pipeline chọn ID một lần rồi dùng cùng manifest cho cả ba roots và mọi representation.

`pilot` chỉ để kiểm tra layout/GPU/độ ổn định code. Hash ranking độc lập theo cell tạo subset lồng nhau; kết quả pilot bị gắn nhãn, không trộn với full run.

In [ ]:
# Stage 3 — deterministically select the common IDs once for all three modalities

def select_rows(rows):
    if RUN_MODE == "full":
        selected = list(rows)
    else:
        limit = int(PILOT_PER_CELL)
        grouped = defaultdict(list)
        for row in rows:
            cell = (row["split"], row["generator"])
            key = f"{SEED}\\0{cell[0]}\\0{cell[1]}\\0{row['sample_id']}".encode()
            rank = hashlib.sha256(key).hexdigest()
            grouped[cell].append((rank, row["sample_id"], row))
        selected = []
        for cell, values in sorted(grouped.items()):
            selected.extend(item[2] for item in sorted(values, key=lambda x: (x[0], x[1]))[:limit])
    selected.sort(key=lambda r: (r["split"], r["generator"], r["sample_id"]))
    counts = defaultdict(int)
    for row in selected:
        counts[f"{row['split']}/{row['generator']}"] += 1
    payload = {"mode": RUN_MODE, "per_cell_cap": None if RUN_MODE == "full" else PILOT_PER_CELL,
               "seed": SEED, "counts": dict(sorted(counts.items())),
               "manifest_sha256": manifest_payload["manifest_sha256"],
               "selected_ids": [{k: r[k] for k in ("split", "generator", "sample_id")} for r in selected]}
    SELECTED_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return selected, payload


selected_rows, selection_payload = select_rows(all_triplets)
print("Selected mode:", RUN_MODE, "rows:", len(selected_rows), "cells:", selection_payload["counts"])
full_counts = defaultdict(int)
for row in all_triplets:
    full_counts[f"{row['split']}/{row['generator']}"] += 1
if RUN_MODE == "pilot":
    ratios = {cell: round(selection_payload["counts"].get(cell, 0) / n, 4)
              for cell, n in sorted(full_counts.items()) if n}
    print("Pilot/full per-cell retained fraction:", ratios,
          "(I/O and epoch sample count scale approximately with retained rows; fixed model overhead remains.)")
else:
    print("Full mode uses every matched protocol ID; this is the recommended result set when compute allows.")
print("Selection manifest:", SELECTED_PATH)

# Tiny decode/shape spot-check only: one matched triple per cell; no full image audit.
from PIL import Image
seen_cells = set()
for row in selected_rows:
    cell = (row["split"], row["generator"])
    if cell in seen_cells:
        continue
    seen_cells.add(cell)
    for asset in ("images", "recons", "dire"):
        with Image.open(row[f"{asset}_path"]) as im:
            im.convert("RGB").load()
            if im.size != (256, 256):
                raise ValueError(f"Expected 256x256, got {im.size}: {row[f'{asset}_path']}")
print(f"Spot-check decoded {len(seen_cells)} cells × 3 modalities; remaining images are checked lazily by the loader.")


## Stage 4 — lazy input views and model protocol

No `.npy` feature cache is created. `RGB`, `REC`, `DIRE` and `RGB_DIRE` are generated in memory from the same three path columns. Training/evaluation crops and flips are synchronized across RGB and DIRE branches.

In [ ]:
# Stage 4 — lazy Dataset, synchronized transform, ResNet-50 ImageNet init
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.transforms import functional as TF
from torchvision.models import resnet50

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def image_tensor(path):
    with Image.open(path) as im:
        return TF.to_tensor(im.convert("RGB"))


def transform_pair(rgb_path, dire_path, training):
    with Image.open(rgb_path) as source:
        rgb = source.convert("RGB")
    if dire_path:
        with Image.open(dire_path) as source:
            dire = source.convert("RGB")
    else:
        dire = None
    resize = T.Resize(256, interpolation=T.InterpolationMode.BILINEAR)
    rgb = resize(rgb)
    if dire is not None:
        dire = resize(dire)
        if rgb.size != dire.size:
            raise ValueError(f"RGB/DIRE shape mismatch: {rgb.size} vs {dire.size}")
    if training:
        top, left, height, width = T.RandomCrop.get_params(rgb, output_size=(224, 224))
        rgb = TF.crop(rgb, top, left, height, width)
        if dire is not None:
            dire = TF.crop(dire, top, left, height, width)
        if random.random() < 0.5:
            rgb = TF.hflip(rgb)
            if dire is not None:
                dire = TF.hflip(dire)
    else:
        rgb = TF.center_crop(rgb, (224, 224))
        if dire is not None:
            dire = TF.center_crop(dire, (224, 224))
    tensors = [TF.to_tensor(rgb)]
    if dire is not None:
        tensors.append(TF.to_tensor(dire))
    x = torch.cat(tensors, dim=0) if len(tensors) == 2 else tensors[0]
    mean = torch.tensor(MEAN * (x.shape[0] // 3), dtype=x.dtype).view(-1, 1, 1)
    std = torch.tensor(STD * (x.shape[0] // 3), dtype=x.dtype).view(-1, 1, 1)
    return (x - mean) / std


class Table5Dataset(Dataset):
    def __init__(self, rows, representation, training=False):
        self.rows = list(rows)
        self.representation = representation
        self.training = training
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        rep = self.representation
        if rep == "RGB":
            x = transform_pair(row["images_path"], None, self.training)
        elif rep == "REC":
            x = transform_pair(row["recons_path"], None, self.training)
        elif rep == "DIRE":
            x = transform_pair(row["dire_path"], None, self.training)
        elif rep == "RGB_DIRE":
            x = transform_pair(row["images_path"], row["dire_path"], self.training)
        else:
            raise ValueError(rep)
        return x, torch.tensor(row["label"], dtype=torch.float32), row["sample_id"]


def load_imagenet_state(path):
    if not path:
        raise FileNotFoundError("Set IMAGENET_CHECKPOINT to a local Kaggle input checkpoint before training")
    p = Path(path).expanduser().resolve()
    if not p.is_file():
        raise FileNotFoundError(f"ImageNet checkpoint missing: {p}")
    try:
        raw = torch.load(p, map_location="cpu", weights_only=True)
    except TypeError:
        raw = torch.load(p, map_location="cpu")
    for key in ("state_dict", "model", "model_state_dict"):
        if isinstance(raw, dict) and key in raw and isinstance(raw[key], dict):
            raw = raw[key]
            break
    if not isinstance(raw, dict):
        raise TypeError("Expected a torchvision ResNet state_dict")
    if raw and all(k.startswith("module.") for k in raw):
        raw = {k[len("module."):]: v for k, v in raw.items()}
    if "conv1.weight" not in raw:
        raise KeyError("ImageNet checkpoint must have torchvision key conv1.weight")
    return raw, p


def new_resnet50(representation, imagenet_state):
    model = resnet50(weights=None)
    if representation == "RGB_DIRE":
        old = model.conv1
        new = nn.Conv2d(6, old.out_channels, old.kernel_size, old.stride, old.padding,
                        bias=False, dilation=old.dilation, groups=old.groups)
        model.conv1 = new
    missing, unexpected = model.load_state_dict(imagenet_state, strict=False)
    missing = set(missing); unexpected = set(unexpected)
    if missing != {"fc.weight", "fc.bias"} or unexpected:
        raise RuntimeError(f"Unexpected checkpoint mapping: missing={missing}, unexpected={unexpected}")
    if representation == "RGB_DIRE":
        with torch.no_grad():
            w = imagenet_state["conv1.weight"]
            model.conv1.weight.copy_(torch.cat((w * 0.5, w * 0.5), dim=1))
    model.fc = nn.Linear(2048, 1)
    nn.init.normal_(model.fc.weight, mean=0.0, std=0.02)
    nn.init.zeros_(model.fc.bias)
    return model

print("Device:", DEVICE, "· ImageNet checkpoint path:", IMAGENET_CHECKPOINT)
print("Input tensors: RGB/REC/DIRE=3ch; RGB_DIRE=6ch; normalization=ImageNet per 3ch group")

## Stage 5 — train and evaluate all four representations

Train/val theo sample IDs của manifest; bốn model chạy tuần tự để không giữ nhiều ResNet trên GPU. Early stopping/giảm LR dựa trên validation ACC và test chỉ chấm sau khi chọn checkpoint. Test ACC dùng strict `score > 0.5` như code tác giả; AP dùng sklearn raw-prevalence AP. RGB+DIRE six-channel initialization là lựa chọn ghi rõ của notebook, chưa xác minh từ code tác giả.

In [ ]:
# Stage 5 — metric functions + source-style training/resume
from sklearn.metrics import average_precision_score, roc_auc_score, confusion_matrix

TRAIN_ROWS = [r for r in selected_rows if r["split"] == "train"]
VAL_ROWS = [r for r in selected_rows if r["split"] == "val"]
TEST_ROWS = [r for r in selected_rows if r["split"] == "test"]


def check_binary_cells(rows, split):
    real = sum(r["generator"] == "real" for r in rows)
    fake = sum(r["generator"] == "adm" for r in rows)
    if real == 0 or fake == 0:
        raise RuntimeError(f"{split} missing real/ADM: real={real}, adm={fake}")
    if real != fake:
        print(f"WARNING: {split} is imbalanced (real={real}, ADM={fake}); "
              "metrics will reflect this sampling prevalence.")
    return real, fake

TRAIN_COUNTS = check_binary_cells(TRAIN_ROWS, "train")
VAL_COUNTS = check_binary_cells(VAL_ROWS, "val")


def eval_binary(model, loader):
    model.eval()
    scores, labels, ids = [], [], []
    with torch.inference_mode():
        for x, y, sample_ids in loader:
            p = torch.sigmoid(model(x.to(DEVICE))).flatten().cpu().numpy()
            scores.extend(map(float, p)); labels.extend(map(int, y.numpy())); ids.extend(sample_ids)
    pred = np.asarray(scores) > 0.5
    y = np.asarray(labels, dtype=np.int64)
    return {"scores": scores, "labels": labels, "sample_ids": ids,
            "acc_at_0_5": float(np.mean(pred == y)),
            "ap": float(average_precision_score(y, scores)),
            "auroc": float(roc_auc_score(y, scores)),
            "confusion_tn_fp_fn_tp": confusion_matrix(y, pred.astype(int), labels=[0, 1]).ravel().tolist()}


def atomic_torch_save(payload, path):
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp); os.replace(tmp, path)


def train_one(representation, imagenet_state):
    seed_everything(SEED)
    out = RESULTS_PATH / representation
    out.mkdir(parents=True, exist_ok=True)
    config = {"representation": representation, "run_mode": RUN_MODE,
              "selected_manifest": str(SELECTED_PATH), "seed": SEED,
              "epochs": MAX_EPOCHS, "batch": BATCH_SIZE, "lr": LEARNING_RATE,
              "patience": PATIENCE, "selection": "source-style val ACC, threshold strict > 0.5",
              "fusion": "channel concat RGB then DIRE; conv1 duplicated and halved (repo choice)",
              "dire_decode_mode": DIRE_DECODE_MODE}
    config_hash = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()
    config_path = out / "protocol.json"
    if config_path.exists() and json.loads(config_path.read_text()).get("config_hash") != config_hash:
        raise RuntimeError(f"Protocol changed in existing result folder {out}; choose a new RUN_ID")
    config_path.write_text(json.dumps({**config, "config_hash": config_hash}, indent=2))

    model = new_resnet50(representation, imagenet_state).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.999))
    loss_fn = nn.BCEWithLogitsLoss()
    start_epoch, best_score, bad_epochs, delta, lr_stage = 0, None, 0, -0.001, 0
    resume_path, best_path = out / "last_resume.pt", out / "best_model.pt"
    history_path = out / "history.json"
    history = json.loads(history_path.read_text()) if history_path.exists() else []
    if resume_path.exists():
        state = torch.load(resume_path, map_location="cpu", weights_only=False)
        if state.get("config_hash") != config_hash:
            raise RuntimeError("Resume state belongs to a different experiment config")
        model.load_state_dict(state["model"]); optimizer.load_state_dict(state["optimizer"])
        start_epoch = int(state["epoch"]); best_score = state["best_score"]
        bad_epochs = int(state["bad_epochs"]); delta = float(state["delta"]); lr_stage = int(state["lr_stage"])
        for group in optimizer.param_groups:
            group["lr"] = float(state["lr"])
        print(f"{representation}: resume from epoch {start_epoch}")

    for epoch in range(start_epoch, MAX_EPOCHS):
        seed_everything(SEED + epoch)
        train_subset = [r for r in TRAIN_ROWS if r["generator"] in ("real", "adm")]
        train_ds = Table5Dataset(train_subset, representation, training=True)
        generator = torch.Generator().manual_seed(SEED + epoch)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
                                  generator=generator, persistent_workers=NUM_WORKERS > 0)
        model.train(); loss_sum = 0.0; n_seen = 0
        for x, y, _ in train_loader:
            x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x).squeeze(1)
            loss = loss_fn(logits, y)
            loss.backward(); optimizer.step()
            loss_sum += float(loss.item()) * len(y); n_seen += len(y)
        val_loader = DataLoader(Table5Dataset(VAL_ROWS, representation, training=False),
                                batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                                pin_memory=(DEVICE.type == "cuda"), persistent_workers=NUM_WORKERS > 0)
        val = eval_binary(model, val_loader)
        val_acc = val["acc_at_0_5"]
        row = {"epoch": epoch + 1, "train_loss": loss_sum / max(n_seen, 1),
               "val_acc_at_0_5": val_acc, "val_ap": val["ap"], "lr": optimizer.param_groups[0]["lr"]}
        history.append(row); history_path.write_text(json.dumps(history, indent=2))

        # Authors' EarlyStopping(delta=-0.001): require a >0.001 ACC gain to reset.
        improved = best_score is None or val_acc >= best_score - delta
        if improved:
            best_score = val_acc; bad_epochs = 0
            atomic_torch_save({"model": model.state_dict(), "epoch": epoch + 1,
                               "val_acc": val_acc, "val_ap": val["ap"], "config_hash": config_hash}, best_path)
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                new_lr = optimizer.param_groups[0]["lr"] / 10.0
                if new_lr < 1e-6:
                    print(f"{representation}: stop at epoch {epoch+1}; LR floor reached")
                    break
                for group in optimizer.param_groups: group["lr"] = new_lr
                lr_stage += 1; delta = -0.002; bad_epochs = 0
                best_score = None  # mirrors the source EarlyStopping reset after LR drop
        if (epoch + 1) % SAVE_RESUME_EVERY == 0:
            atomic_torch_save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                               "epoch": epoch + 1, "best_score": best_score,
                               "bad_epochs": bad_epochs, "delta": delta, "lr_stage": lr_stage,
                               "lr": optimizer.param_groups[0]["lr"], "config_hash": config_hash}, resume_path)
        print(f"{representation} epoch {epoch+1:03d}/{MAX_EPOCHS}: loss={row['train_loss']:.5f} "
              f"val_acc={val_acc:.4f} val_ap={val['ap']:.5f} lr={row['lr']:.1e}")

    if not best_path.exists():
        raise RuntimeError(f"No best checkpoint was written for {representation}")
    best = torch.load(best_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(best["model"]); model.eval()
    return model, {"best_epoch": best["epoch"], "best_val_acc": best["val_acc"],
                   "best_val_ap": best["val_ap"], "epochs_recorded": len(history),
                   "checkpoint": str(best_path), "config": config}


## Stage 6 — run all models once the preflight is acceptable

Khi bật `CONFIRM_GPU_RUN=True`, cell này train bốn model tuần tự, chọn checkpoint từ validation, chấm test từng generator, ghi scores/metrics/provenance trong `/kaggle/working`. Notebook không ghi vào `/kaggle/input`, không tạo bản sao dữ liệu và không tự upload artifact lên Kaggle.

In [ ]:
# Stage 6 — one run-all entrypoint: preflight -> train four reps -> evaluate -> report

def evaluate_test_matrix(model, representation):
    report = {}
    score_rows = []
    real_rows = [r for r in TEST_ROWS if r["generator"] == "real"]
    for generator in TEST_GENERATORS:
        fake_rows = [r for r in TEST_ROWS if r["generator"] == generator]
        if not real_rows or not fake_rows:
            raise RuntimeError(f"Missing test cell {generator}: real={len(real_rows)} fake={len(fake_rows)}")
        cell_rows = real_rows + fake_rows
        ds = Table5Dataset(cell_rows, representation, training=False)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                            pin_memory=(DEVICE.type == "cuda"), persistent_workers=NUM_WORKERS > 0)
        metrics = eval_binary(model, loader)
        report[generator] = {"status": "ran", "n_real": len(real_rows), "n_fake": len(fake_rows),
                             "acc_at_0_5": metrics["acc_at_0_5"], "ap": metrics["ap"],
                             "auroc": metrics["auroc"],
                             "confusion_tn_fp_fn_tp": metrics["confusion_tn_fp_fn_tp"]}
        for sample_id, label, score in zip(metrics["sample_ids"], metrics["labels"], metrics["scores"]):
            score_rows.append({"representation": representation, "generator": generator,
                               "sample_id": sample_id, "label": label, "score_fake": score})
    with (RESULTS_PATH / representation / "scores.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["representation", "generator", "sample_id", "label", "score_fake"])
        writer.writeheader(); writer.writerows(score_rows)
    return report


def run_all():
    if not CONFIRM_GPU_RUN:
        print("Preflight/index/subset are ready. Training is deliberately disabled. "
              "Set CONFIRM_GPU_RUN=True only after reviewing counts, SD-v1/SD-v2 scope, checkpoint, and Kaggle quota.")
        return None
    if DEVICE.type != "cuda":
        raise RuntimeError("GPU was explicitly requested but CUDA is unavailable")
    state, checkpoint_path = load_imagenet_state(IMAGENET_CHECKPOINT)
    ckpt_sha = hashlib.sha256(Path(checkpoint_path).read_bytes()).hexdigest()
    for split in ("train", "val"):
        check_binary_cells([r for r in selected_rows if r["split"] == split], split)
    results = {"status": "running", "run_id": RUN_ID, "protocol_label": PROTOCOL_LABEL,
               "run_mode": RUN_MODE, "seed": SEED, "domain": DOMAIN,
               "representations": REPRESENTATIONS, "test_generators": TEST_GENERATORS,
               "dataset_version_tag": DATASET_VERSION_TAG,
               "triplet_manifest_sha256": manifest_payload["manifest_sha256"],
               "selection": selection_payload, "counts": selection_payload["counts"],
               "imagenet_checkpoint": str(checkpoint_path), "imagenet_checkpoint_sha256": ckpt_sha,
               "dire_decode_mode": DIRE_DECODE_MODE,
               "limitations": ["DIRE/recons reused from mounted release; no clean-room map recompute.",
                               "RGB_DIRE concat/6-channel conv1 init is the notebook's recorded choice, not author-verified.",
                               "If SD-v2 is selected, it is a variant and not SD-v1 paper reproduction."]}
    RESULTS_PATH.mkdir(parents=True, exist_ok=True)
    (RESULTS_PATH / "run_protocol.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
    for rep in REPRESENTATIONS:
        print(f"\\n===== {rep}: train and test =====")
        model, train_meta = train_one(rep, state)
        results.setdefault("by_representation", {})[rep] = {
            "train": train_meta, "test": evaluate_test_matrix(model, rep)}
        (RESULTS_PATH / "run_protocol.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
        del model
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
    results["status"] = "complete"
    (RESULTS_PATH / "run_protocol.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
    return results


RUN_RESULTS = run_all()
if RUN_RESULTS is None:
    print("No GPU work started. Selected manifest remains at", SELECTED_PATH)
else:
    print("Completed. Machine-readable report:", RESULTS_PATH / "run_protocol.json")

## Stage 7 — final checks and deliverables

`results/run_protocol.json` chứa config, counts, hash manifest/checkpoint, metric mỗi representation × generator và quyết định protocol. Mỗi `scores.csv` giữ sample ID, nhãn và score để tính lại bảng. Checkpoint và history nằm trong thư mục từng representation. Không xem kết quả pilot là paper result; không đổi generator alias hoặc chỉnh threshold theo test.

### Bước của bạn sau khi notebook được attach

1. Attach ba Dataset inputs và ImageNet checkpoint; nếu auto-discovery mơ hồ thì điền ba paths chính xác ở Stage 0.
2. Chạy **Run All** ở `RUN_MODE="full"` sau khi xác nhận test generator matrix. Nếu chỉ có SD-v2, chọn variant tường minh.
3. Đọc counts / manifest trước khi bật training; sau đó đặt `CONFIRM_GPU_RUN=True` và chạy Run All.
4. Download/Version outputs của Kaggle session để lưu bền vững. Input vẫn read-only.
